# Sparse Retrieval: Inverted Indexes and BM25

Dense retrieval represents text with learned vectors and can match related meanings across different wording. Sparse retrieval represents text through terms and their weights. It is especially useful when the query contains exact names, product codes, rare terms, or wording that should match the source directly.

This notebook builds a small inverted index from scratch and uses it to retrieve chunks with BM25, a widely used lexical ranking function. The implementation uses only Python's standard library so you can inspect every calculation. It is a learning implementation, not a production search engine.

## Learning goals

By the end, you should be able to:

- Explain sparse term representations, document frequency, and an inverted index.
- Trace a query term from tokenization to matching chunks.
- Explain BM25's inverse document frequency, term-frequency saturation, and document-length normalization.
- Implement an auditable BM25 scorer and rank top-K chunks deterministically.
- Compare BM25 with simple overlap and dense retrieval at a conceptual level.
- Identify lexical retrieval's strengths and limitations and evaluate results with relevance labels.

## 1. Where sparse retrieval fits

The retrieval pipeline is still **query → candidates → scores → ranking → top-K**. The difference is the evidence used to score a query–chunk pair.

- **Dense retrieval** compares learned vector representations. Related wording can match even without shared terms.
- **Sparse retrieval** matches query terms to terms in the corpus and weights those matches. Unmatched terms generally contribute no evidence.
- **BM25** is a strong, practical lexical scoring function that improves on counting overlapping words.

The term “sparse” describes a representation where only a small subset of possible vocabulary terms is present in any particular text. It does not mean the search is crude or unimportant. Lexical retrieval is often excellent for exact terminology and complements semantic search in hybrid systems.

### From a bag of words to an inverted index

A simple term-based representation can be imagined as one coordinate per vocabulary term. Most documents use only a tiny fraction of the vocabulary, so most coordinates would be zero. Instead of storing a huge mostly-empty vector for every chunk, search engines commonly use an **inverted index**:

`term → documents/chunks that contain the term (and often where/how often it occurs)`

For example, `helmet → [C07]` and `account → [C08, C10]` in our small corpus. At query time, the index can look up query terms and gather only chunks containing at least one of them. The postings list can store term frequency and other positional information as well.

## 2. Reuse the teaching corpus

We keep the same chunk IDs and content used in the previous retrieval lessons. In production, these chunk records would come from document ingestion and chunking, and their terms would be indexed when content is added or updated.

In [1]:
corpus = [
    {'chunk_id': 'C01', 'source': 'account_help', 'topic': 'password',
     'text': 'To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.'},
    {'chunk_id': 'C02', 'source': 'account_help', 'topic': 'security',
     'text': 'For account security, turn on two-factor authentication and never share your verification code with anyone.'},
    {'chunk_id': 'C03', 'source': 'shopping_policy', 'topic': 'refunds',
     'text': 'Refunds for eligible orders are returned to the original payment method after the returned item is inspected.'},
    {'chunk_id': 'C04', 'source': 'shipping_guide', 'topic': 'delivery',
     'text': 'Standard shipping usually takes five to seven business days. Express shipping is available at checkout.'},
    {'chunk_id': 'C05', 'source': 'shopping_policy', 'topic': 'returns',
     'text': 'Unopened products can be returned within thirty days of delivery. Include the order number in your return request.'},
    {'chunk_id': 'C06', 'source': 'tenant_handbook', 'topic': 'lease',
     'text': 'The security deposit is due when the lease is signed. The deposit amount is listed in the signed lease agreement.'},
    {'chunk_id': 'C07', 'source': 'cycling_guide', 'topic': 'safety',
     'text': 'Choose a bicycle helmet that fits snugly and sits level on the head. Replace a helmet after a serious impact.'},
    {'chunk_id': 'C08', 'source': 'account_help', 'topic': 'support',
     'text': 'Contact support if you cannot access the email address linked to your account.'},
    {'chunk_id': 'C09', 'source': 'food_guide', 'topic': 'allergies',
     'text': 'Read the ingredient label carefully if you have a food allergy. Contact the manufacturer when allergen information is unclear.'},
    {'chunk_id': 'C10', 'source': 'account_help', 'topic': 'limitations',
     'text': 'Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.'},
]

print(f'{len(corpus)} chunks in the teaching corpus.')
for chunk in corpus[:3]:
    print(chunk['chunk_id'], chunk['topic'], '|', chunk['text'])

10 chunks in the teaching corpus.
C01 password | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
C02 security | For account security, turn on two-factor authentication and never share your verification code with anyone.
C03 refunds | Refunds for eligible orders are returned to the original payment method after the returned item is inspected.


## 3. Tokenize and build the inverted index

For a transparent baseline, we lowercase text and extract letter-and-digit sequences. We do not remove stop words or stem words in this first pass. Therefore `return` and `returned` remain different terms, as do `helmet` and `helmets`.

The inverted index maps each term to a **postings list** of chunk IDs. We also store term frequencies per chunk so BM25 can distinguish one occurrence from repeated occurrences. This toy index stores IDs in deterministic corpus order; production engines use optimized data structures, compression, and often positional postings.

In [2]:
import math
import re
from collections import Counter, defaultdict

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

doc_term_frequencies = {}
inverted_index = defaultdict(list)

for chunk in corpus:
    terms = tokenize(chunk['text'])
    term_frequencies = Counter(terms)
    doc_term_frequencies[chunk['chunk_id']] = term_frequencies
    for term in sorted(term_frequencies):
        inverted_index[term].append(chunk['chunk_id'])

print('Vocabulary size:', len(inverted_index))
for term in ('password', 'account', 'helmet', 'nonexistent'):
    print(f'{term!r} -> {inverted_index.get(term, [])}')

Vocabulary size: 122
'password' -> ['C01', 'C10']
'account' -> ['C02', 'C08', 'C10']
'helmet' -> ['C07']
'nonexistent' -> []


A postings list includes a chunk once, even if that term occurs several times; its frequency is available separately. An actual index can use that frequency directly in postings, avoiding a second lookup. The distinction matters: **document frequency** counts how many chunks contain a term, while **term frequency** counts how many times that term appears within one particular chunk.

In [3]:
def document_frequency(term):
    return len(inverted_index.get(term, []))

for term in ('password', 'account', 'helmet', 'the'):
    print(f"df({term!r}) = {document_frequency(term)} chunks")

example_id = 'C01'
print("\nC01 term frequency for 'password':", doc_term_frequencies[example_id]['password'])

df('password') = 2 chunks
df('account') = 3 chunks
df('helmet') = 1 chunks
df('the') = 7 chunks

C01 term frequency for 'password': 2


## 4. Candidate generation from postings

With an inverted index, one simple candidate strategy is to take the union of postings for the query's terms. A chunk with no query terms is not considered. This is only one possible candidate strategy; engines may use Boolean constraints, field-specific postings, phrase/proximity operators, or other methods.

This creates an important boundary: the candidate set can affect what a later ranking stage can return. If the query shares no indexed terms with a relevant chunk, this simple lexical candidate generator will not include it—even if a human sees a semantic connection.

In [4]:
def lexical_candidates(query_text):
    candidates = set()
    for term in set(tokenize(query_text)):
        candidates.update(inverted_index.get(term, []))
    return sorted(candidates)

candidate_query = 'login secret regain entry'
candidates = lexical_candidates(candidate_query)
print('Query tokens:', sorted(set(tokenize(candidate_query))))
print('Candidate chunk IDs:', candidates)
print('Relevant password instructions C01 included?', 'C01' in candidates)

Query tokens: ['entry', 'login', 'regain', 'secret']
Candidate chunk IDs: []
Relevant password instructions C01 included? False


The paraphrase does not contain any terms in the corpus for the password-reset concept. This simple lexical candidate generator therefore cannot find `C01` for the paraphrase. It is not a bug in the index: it reflects the method's dependence on shared terms. Dense retrieval or query expansion can address some vocabulary mismatch.

## 5. Why not just count matching words?

A simple overlap score gives one point for every distinct query term found in a chunk. It is a useful baseline, but it treats all terms equally and ignores how often a term appears in the corpus. A rare, distinctive term such as `helmet` may be more informative than a common term such as `the`.

BM25 improves lexical scoring with three main ideas:

1. **Inverse document frequency (IDF):** terms found in fewer chunks receive more weight than terms found in many chunks.
2. **Term-frequency saturation:** additional repetitions can help, but their benefit diminishes rather than growing without bound.
3. **Document-length normalization:** a long chunk has more opportunities to contain query terms, so BM25 adjusts for length relative to the corpus average.

BM25 is a family of ranking functions with common parameterizations. The next cell implements a standard Okapi BM25-style formula using a nonnegative IDF variant. Libraries may differ in details, so scores from different implementations should not be assumed identical.

### BM25 formula used here

For query term `t` and chunk `D`, this notebook uses:

`score(D, Q) = Σ IDF(t) × [ f(t,D) × (k1 + 1) ] / [ f(t,D) + k1 × (1 - b + b × |D| / avgdl) ]`

where:

- `f(t,D)` is the term frequency of `t` in chunk `D`.
- `|D|` is the chunk length in tokens and `avgdl` is average chunk length.
- `k1` controls term-frequency saturation (often a value around 1.2–2 is used as a starting point).
- `b` controls length normalization, from 0 (none) to 1 (full normalization by length relative to average).
- `IDF(t) = log(1 + (N - df(t) + 0.5) / (df(t) + 0.5))`, where N is the number of chunks and df is the number containing the term.

This IDF variant is positive, including for common terms. Other BM25 formulations use different IDF expressions and may handle negative IDF differently. The formula, tokenization, corpus statistics, and parameters together define the score.

In [5]:
N = len(corpus)
doc_lengths = {
    chunk['chunk_id']: len(tokenize(chunk['text']))
    for chunk in corpus
}
average_doc_length = sum(doc_lengths.values()) / N

print('Number of chunks N:', N)
print('Average chunk length:', round(average_doc_length, 2), 'tokens')
print('C01 length:', doc_lengths['C01'], 'tokens')

Number of chunks N: 10
Average chunk length: 18.0 tokens
C01 length: 22 tokens


### Inspect inverse document frequency

A term that appears in fewer chunks receives a higher IDF under this formula. It is a corpus-level weight: it depends on how many chunks contain a term, not how many times it appears in one particular chunk. Rare does not automatically mean relevant—a rare typo or irrelevant identifier can still have a high IDF.

In [6]:
def idf(term):
    df = document_frequency(term)
    return math.log(1 + (N - df + 0.5) / (df + 0.5))

for term in ('password', 'account', 'helmet', 'the', 'not_in_vocabulary'):
    print(f"IDF({term!r}) = {idf(term):.3f}; df = {document_frequency(term)}")

assert idf('helmet') > idf('account')
assert idf('not_in_vocabulary') > 0

IDF('password') = 1.482; df = 2
IDF('account') = 1.145; df = 3
IDF('helmet') = 1.992; df = 1
IDF('the') = 0.383; df = 7
IDF('not_in_vocabulary') = 3.091; df = 0


The unknown term has df=0 and a positive IDF from the smoothing formula, but it contributes no BM25 score to any chunk because no chunk has a matching term frequency. IDF alone never creates a match; query terms still need postings in a candidate chunk.

## 6. Implement BM25 scoring

The implementation below scores unique query terms. Repeating a term in the query does not add repeated contributions in this simplified setup. Some BM25 implementations include query-term-frequency weighting; this notebook omits it to focus on document-side term frequency, IDF, and length normalization.

In [7]:
def bm25_score(query_text, chunk_id, k1=1.5, b=0.75):
    if k1 < 0:
        raise ValueError('k1 must be non-negative')
    if not 0 <= b <= 1:
        raise ValueError('b must be between 0 and 1')

    query_terms = set(tokenize(query_text))
    term_frequencies = doc_term_frequencies[chunk_id]
    doc_length = doc_lengths[chunk_id]
    length_ratio = doc_length / average_doc_length if average_doc_length else 0.0

    score = 0.0
    for term in query_terms:
        frequency = term_frequencies.get(term, 0)
        if frequency == 0:
            continue
        denominator = frequency + k1 * (1 - b + b * length_ratio)
        score += idf(term) * (frequency * (k1 + 1)) / denominator
    return score

bm25_query = 'password reset link'
for chunk in corpus:
    score = bm25_score(bm25_query, chunk['chunk_id'])
    if score > 0:
        print(chunk['chunk_id'], round(score, 3), '|', chunk['text'])

C01 5.762 | To reset your password, choose Forgot password on the sign-in page. We send a reset link to your registered email address.
C10 2.822 | Archived note: it describes how password reset worked in an old unrelated system, but gives no current account recovery instructions.


Each matching query term contributes an IDF-weighted, length-adjusted amount. If a term occurs more often in a chunk, its contribution rises but saturates. The calculation is deterministic for a fixed corpus, tokenizer, query, and parameter set.

### Term-frequency saturation

The next table holds corpus statistics fixed and varies the frequency of one term in a hypothetical chunk of average length. The first few occurrences increase the term contribution; later occurrences provide less additional value. Repeating a word endlessly should not make a passage infinitely relevant.

In [ ]:
def bm25_term_component(term, frequency, doc_length, k1=1.5, b=0.75):
    if frequency <= 0:
        return 0.0
    ratio = doc_length / average_doc_length if average_doc_length else 0.0
    denominator = frequency + k1 * (1 - b + b * ratio)
    return idf(term) * (frequency * (k1 + 1)) / denominator

for frequency in (1, 2, 4, 8, 16):
    value = bm25_term_component('password', frequency, average_doc_length)
    print(f'tf={frequency:2d} -> contribution={value:.3f}')

### Document-length normalization

For the same term frequency, a match in a much longer chunk is generally less distinctive than a match in a short chunk. Parameter `b` controls the strength of this adjustment: at `b=0`, chunk length does not affect the denominator; at `b=1`, the denominator uses the full length ratio. Neither extreme is automatically best. Production values should be selected for the search task and chunking style.

In [ ]:
for length in (max(1, int(average_doc_length / 2)), int(average_doc_length), int(average_doc_length * 2)):
    scores = {
        b_value: bm25_term_component('password', frequency=1, doc_length=length, b=b_value)
        for b_value in (0.0, 0.75, 1.0)
    }
    print(f'length={length:2d}:', {b_value: round(value, 3) for b_value, value in scores.items()})

Our chunks are all fairly short and similar in length, so length normalization may have only a modest effect in this toy corpus. It becomes more consequential when document or chunk lengths vary substantially. Chunking design therefore influences BM25 statistics as well as dense embeddings.

## 7. Rank results and return top-K

A chunk that shares no query terms receives a score of zero. For candidate generation, we return only chunks with at least one matched term (positive score), then rank by BM25 score descending with a deterministic chunk-ID tie-breaker. K limits the returned list just as in the preceding notebooks.

In [ ]:
def bm25_rank(query_text, chunks, k1=1.5, b=0.75):
    ranked = []
    for chunk in chunks:
        score = bm25_score(query_text, chunk['chunk_id'], k1=k1, b=b)
        if score > 0:
            ranked.append({**chunk, 'score': score})
    return sorted(ranked, key=lambda item: (-item['score'], item['chunk_id']))

def bm25_retrieve(query_text, chunks, k=3, k1=1.5, b=0.75):
    if k < 0:
        raise ValueError('k must be non-negative')
    return bm25_rank(query_text, chunks, k1=k1, b=b)[:k]

query = 'password reset link'
ranked_results = bm25_rank(query, corpus)
for rank, item in enumerate(ranked_results, start=1):
    print(f"{rank}. {item['chunk_id']} BM25={item['score']:.3f} | {item['text']}")

print('Top 1:', [item['chunk_id'] for item in bm25_retrieve(query, corpus, k=1)])

The archived note `C10` may score highly because it repeats the query terms, although the text says it offers no current instructions. BM25 weights term matches more thoughtfully than overlap counting, but it remains lexical: it does not understand that a passage is out of date or insufficient merely because its terms match.

### BM25 and simple overlap side by side

The earlier overlap score counted distinct matching terms equally. Here we rank using the same tokenization so we can isolate how IDF, term frequency, and length normalization change the scoring behavior. Neither ordering is guaranteed to be correct; inspect the actual passages and compare against relevance labels.

In [ ]:
def overlap_rank(query_text, chunks):
    query_terms = set(tokenize(query_text))
    results = []
    for chunk in chunks:
        score = len(query_terms & set(tokenize(chunk['text'])))
        if score > 0:
            results.append({**chunk, 'score': score})
    return sorted(results, key=lambda item: (-item['score'], item['chunk_id']))

overlap_results = overlap_rank(query, corpus)
bm25_results = bm25_rank(query, corpus)

print('Overlap ranking:')
for item in overlap_results:
    print(f"  {item['chunk_id']} overlap={item['score']} | {item['text']}")

print('\nBM25 ranking:')
for item in bm25_results:
    print(f"  {item['chunk_id']} BM25={item['score']:.3f} | {item['text']}")

## 8. Queries that show strengths and limitations

BM25 is often effective when the query uses important terms from the source. It can struggle with paraphrases, synonyms, morphology, and implied meaning unless a system adds normalization, expansion, or another retrieval method. Try these examples and inspect results rather than looking only at scores.

In [ ]:
queries = [
    'password reset link',
    'login secret regain entry',
    'return period unopened products',
    'when due security deposit lease',
    'bicycle helmet impact',
    'SKU ZX-4817 replacement cable',
]

for query_text in queries:
    results = bm25_retrieve(query_text, corpus, k=3)
    print(f'\nQuery: {query_text}')
    if not results:
        print('  No chunks matched the query terms.')
    for item in results:
        print(f"  {item['chunk_id']} BM25={item['score']:.3f} | {item['text']}")

The final example contains an exact-looking product code that is not in the corpus, so no matching chunk should be returned. An inverted index cannot retrieve evidence for a term that was never indexed. BM25 is useful for exact identifiers when those identifiers are present in the source, but it cannot invent missing evidence.

### Rare terms and exact identifiers

Rare terms receive relatively high IDF because they occur in few chunks. That can make domain terms, names, codes, or error strings valuable matching signals. However, an IDF boost is not a quality guarantee: an accidental identifier match, a typo, or an outdated passage can still rank highly. Search relevance depends on whether the matched text serves the query's need.

## 9. Tokenization, stop words, stemming, and phrase matching

Our tokenizer is intentionally limited. Practical search systems make additional choices that affect both the index and the query:

- **Stop words:** very common terms such as “the” may contribute little discrimination. Removing them can reduce noise, but careless removal may harm phrases, names, or domain-specific queries.
- **Stemming or lemmatization:** mapping related word forms such as `returned` and `return` can improve recall, but may conflate distinct terms. Not all BM25 systems stem by default.
- **Punctuation and identifiers:** splitting `ZX-4817`, `C++`, or version strings incorrectly can damage exact matching. Tokenization should reflect the data domain.
- **Phrase and proximity search:** a basic bag-of-words score ignores term order and distance. A phrase-aware index can distinguish “deposit before signing” from “deposit after signing,” though interpretation still requires reading context.
- **Fields and boosts:** title, heading, body, and tags may be indexed separately and weighted differently.

There is no one tokenizer or preprocessing recipe that is correct for every corpus. The query and document text should be normalized consistently, and changes to indexing typically require rebuilding or updating the index.

### A small tokenizer normalization experiment

This optional experiment applies a deliberately tiny suffix rule to demonstrate how normalization can connect word forms. It is **not** a recommended stemmer: linguistic rules have exceptions, and ad hoc stemming can create false matches. Use it to reason about trade-offs, not as production code.

In [ ]:
def toy_stem(token):
    if len(token) > 5 and token.endswith('ed'):
        return token[:-2]
    if len(token) > 5 and token.endswith('ing'):
        return token[:-3]
    if len(token) > 4 and token.endswith('s'):
        return token[:-1]
    return token

examples = ['return', 'returned', 'returns', 'shipping', 'passwords']
print([(word, toy_stem(word)) for word in examples])
print('Base tokens:', tokenize('returned products'))
print('Toy-normalized:', {toy_stem(term) for term in tokenize('returned products')})

Before applying normalization in a real search system, test it against representative queries and relevance labels. Stemming can improve recall for inflected forms, but it may also merge words that should remain distinct. Search libraries offer tested analyzers and language-specific options; their behavior should be documented and evaluated.

## 10. Evaluate sparse retrieval

Use explicit relevance judgments rather than assuming a high BM25 score means a result is useful. We will reuse the password query's teaching labels: `C01` has current reset instructions, while `C10` is related but not sufficient. A strong lexical match to `C10` is a ranking failure for this information need.

In [ ]:
evaluation_query = 'password reset link'
relevant_ids = {'C01'}

def precision_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    selected = results[:k]
    if not selected:
        return 0.0
    hits = sum(item['chunk_id'] in relevant_ids for item in selected)
    return hits / len(selected)

def recall_at_k(results, relevant_ids, k):
    if k <= 0:
        raise ValueError('k must be positive')
    if not relevant_ids:
        raise ValueError('relevant_ids must not be empty')
    hits = sum(item['chunk_id'] in relevant_ids for item in results[:k])
    return hits / len(relevant_ids)

for method, results in [
    ('overlap', overlap_rank(evaluation_query, corpus)),
    ('BM25', bm25_rank(evaluation_query, corpus)),
]:
    print(method, 'ranking:', [item['chunk_id'] for item in results])
    for k in (1, 3, 5):
        print(f'  P@{k}={precision_at_k(results, relevant_ids, k):.2f}, '
              f'R@{k}={recall_at_k(results, relevant_ids, k):.2f}')

The numerical scores and rankings are specific to this tiny corpus and tokenizer. A meaningful evaluation should include many representative queries: exact terms, paraphrases, rare names, misspellings, multi-part questions, and queries with no answer in the collection. Review relevance labels, compare methods at the same K, and inspect the actual retrieved text.

## 11. When sparse retrieval is useful—and when it is not

| Search need | Sparse/BM25 behavior to consider |
| --- | --- |
| Exact product ID, error code, or proper noun | Often strong if tokenization preserves the term. |
| Query uses the same technical vocabulary as the source | Often strong; IDF helps prioritize distinctive terms. |
| Query uses synonyms absent from the source | May miss the passage without expansion or semantic retrieval. |
| Query asks a subtle relation or negation | Term overlap alone may confuse it; phrase-aware retrieval and reranking can help. |
| Corpus contains long and short chunks | Length normalization matters, but chunking still affects scores. |
| No relevant evidence is indexed | Sparse search cannot retrieve what is missing. |

Sparse and dense methods are complementary. Later, hybrid retrieval will combine their evidence—often using rank fusion rather than adding incomparable raw scores—and evaluate whether the combined results improve relevance.

## 12. Exercises

Try these before looking up external search libraries:

1. Calculate the postings list and document frequency for `return`, `returned`, and `the`. Why does this tokenizer treat the first two as different terms?
2. Change `b` from `0.75` to `0` and then `1`. Which rankings change? Explain the length-normalization effect.
3. Create a query containing a rare term from one chunk and a common term from many chunks. Compare their IDF values and explain the ranking.
4. Add a relevant chunk that uses `credentials` where the query uses `password`. Does BM25 find it without a shared term? What approaches could help?
5. Add an exact product code to one chunk and query for that code. Confirm it is retrievable, then alter tokenization to see how punctuation affects it.
6. Build relevance labels for at least five queries and compute precision@K and recall@K for overlap and BM25. Which queries expose a limitation of each method?
7. Explain why `C10` can score well for password-reset terms without being suitable evidence for current instructions.

### Optional extension: change BM25 parameters

Use this cell to compare a few parameter choices. Keep the query and corpus fixed, record the ranked IDs, and avoid deciding that one setting is universally better based on one query.

In [ ]:
parameter_sets = [
    {'k1': 0.5, 'b': 0.75},
    {'k1': 1.5, 'b': 0.0},
    {'k1': 1.5, 'b': 0.75},
    {'k1': 2.0, 'b': 1.0},
]

for parameters in parameter_sets:
    results = bm25_rank(evaluation_query, corpus, **parameters)
    print(parameters, [(item['chunk_id'], round(item['score'], 3)) for item in results])

## Summary

Sparse retrieval matches query terms against indexed corpus terms. An inverted index maps each term to chunks containing it, enabling efficient candidate generation. BM25 improves over raw overlap by weighting terms according to how rare they are, saturating repeated term frequency, and optionally normalizing for chunk length. Its behavior depends on tokenization, corpus statistics, and parameters.

BM25 is a strong choice for shared vocabulary and exact terminology, but it cannot reliably bridge synonyms or infer whether a matching passage truly supports the answer. Scores are not probabilities. Evaluate rankings using judged queries, and inspect failure cases. The next notebook combines dense and sparse retrieval into **hybrid retrieval**, so semantic matches and exact-term matches can complement one another.